# VigiLens: Exploratory Analysis

This beginner-friendly notebook explores the credit card fraud dataset before any modeling. It reads the existing CSV without changing it. The `Class` column uses `0` for non-fraud and `1` for fraud.

The anonymised features `V1`–`V28` are principal component analysis (PCA) features. `Time` is elapsed seconds from the first transaction in the dataset; it is **not** a clock time or a date.

No model is trained in this notebook.

## 0. Load the local dataset

The path is built with `pathlib`. The notebook searches the current directory and its parents for a project `data/` directory, so it works when launched from the repository root or from `ml/notebooks/`. If the CSV is unavailable, the cell stops with instructions for downloading the original dataset. It never creates replacement data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

current_dir = Path.cwd().resolve()
project_root = next(
    (path for path in (current_dir, *current_dir.parents) if (path / "data").is_dir()),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project data/ directory from the notebook's current working directory. "
        "In VS Code, open the VigiLens project folder and select its Python environment kernel, "
        "then run this notebook from the workspace."
    )

dataset_path = project_root / "data" / "creditcard.csv"
if not dataset_path.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {dataset_path}. Download the Credit Card Fraud Detection dataset from "
        "https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud and place the original "
        "creditcard.csv file in the project's data/ folder. No substitute data was generated."
    )

df = pd.read_csv(dataset_path)
required_columns = {"Class", "Amount", "Time"}
missing_required = required_columns.difference(df.columns)
if missing_required:
    raise ValueError(f"The CSV is missing required columns: {sorted(missing_required)}")

unexpected_classes = set(df["Class"].dropna().unique()).difference({0, 1})
if unexpected_classes:
    raise ValueError(f"Expected Class values 0 and 1; found unexpected values: {unexpected_classes}")

print(f"Loaded: {dataset_path}")
print(f"Dataset size in memory: {df.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MiB")

## 1. Dataset shape and first five rows

The shape is reported as `(rows, columns)`. The preview helps us confirm that the file loaded and see a few records; it is not a complete view of the dataset.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())

## 2. Column names and data types

Data types can affect calculations and plots. This table lists every column and its pandas dtype.

In [ ]:
column_info = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
})
display(column_info)

## 3. Missing values and duplicate rows

Missing values are counted per column. Duplicate rows are counted across all columns; this check does not remove or modify any rows.

In [ ]:
missing_by_column = df.isna().sum().sort_values(ascending=False)
print(f"Total missing cells: {int(missing_by_column.sum()):,}")
print(f"Duplicate rows: {int(df.duplicated().sum()):,}")
display(missing_by_column.rename("missing_values").to_frame())

## 4. Fraud class counts and percentages

We explicitly include both expected classes in the output, including a zero count if a class is absent. Percentages use the total number of rows as the denominator.

In [ ]:
class_counts = df["Class"].value_counts().reindex([0, 1], fill_value=0)
class_summary = pd.DataFrame({
    "class_name": ["Non-fraud", "Fraud"],
    "count": [int(class_counts.loc[0]), int(class_counts.loc[1])],
    "percentage": [
        class_counts.loc[0] / len(df) * 100 if len(df) else 0.0,
        class_counts.loc[1] / len(df) * 100 if len(df) else 0.0,
    ],
}, index=pd.Index([0, 1], name="Class"))
display(class_summary.style.format({"percentage": "{:.4f}%"}))

## 5. Class distribution

Fraud datasets can be imbalanced, so the chart keeps both class labels visible and prints each bar's row count. The y-axis uses a logarithmic scale when both classes are present so a small fraud count remains visible alongside the non-fraud count.

In [ ]:
class_plot = class_counts.rename(index={0: "Non-fraud (0)", 1: "Fraud (1)"})
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(class_plot.index, class_plot.values, color=["#4C78A8", "#E45756"])
if (class_plot > 0).all():
    ax.set_yscale("log")
    ax.set_ylabel("Transaction count (log scale)")
else:
    ax.set_ylabel("Transaction count")
ax.set_title("Transactions by class")
ax.set_xlabel("Class")
ax.set_xticks(range(2), class_plot.index)
for bar, count in zip(bars, class_plot.values):
    label_y = count if count > 0 else 0
    ax.annotate(f"{count:,}", (bar.get_x() + bar.get_width() / 2, label_y),
                xytext=(0, 5), textcoords="offset points", ha="center", va="bottom")
ax.set_ylim(bottom=0.1 if (class_plot > 0).all() else 0)
plt.tight_layout()
plt.show()

## 6. Transaction Amount by class

Transaction amounts are often strongly right-skewed. We plot `log1p(Amount)` (the natural logarithm of `1 + Amount`) so zero amounts remain valid and large values do not dominate the view. The density histograms compare the two classes, and the boxplot gives a compact view of their spread. The source `Amount` column is left unchanged.

In [ ]:
amount_plot = df[["Amount", "Class"]].copy()
amount_plot["log1p_amount"] = __import__("numpy").log1p(amount_plot["Amount"])
amount_plot["class_name"] = amount_plot["Class"].map({0: "Non-fraud (0)", 1: "Fraud (1)"})
class_order = ["Non-fraud (0)", "Fraud (1)"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=amount_plot, x="log1p_amount", hue="class_name", hue_order=class_order,
             stat="density", common_norm=False, element="step", fill=False, bins=60, ax=axes[0])
axes[0].set_title("Amount distribution by class")
axes[0].set_xlabel("log1p(Amount)")
axes[0].set_ylabel("Density within class")
sns.boxplot(data=amount_plot, x="class_name", y="log1p_amount", order=class_order, ax=axes[1])
axes[1].set_title("Amount spread by class")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("log1p(Amount)")
plt.tight_layout()
plt.show()

## 7. Transaction Time distribution

`Time` records elapsed seconds relative to the first transaction in this dataset. It does not tell us the time of day. This histogram shows how transactions are distributed over that elapsed interval.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(data=df, x="Time", bins=60, color="#4C78A8", ax=ax)
ax.set_title("Transaction distribution over elapsed time")
ax.set_xlabel("Elapsed seconds from the first transaction (not clock time)")
ax.set_ylabel("Transaction count")
plt.tight_layout()
plt.show()

## 8. Descriptive statistics and correlations with `Class`

The descriptive table summarizes numeric columns. The second table sorts numeric feature correlations with `Class` by absolute magnitude. Correlation is a linear association, not evidence that a feature causes fraud; it also does not establish that a feature will be useful in a future model.

In [ ]:
numeric_df = df.select_dtypes(include="number")
display(numeric_df.describe().T)

correlations_with_class = (
    numeric_df.corr()["Class"]
    .drop(labels=["Class"], errors="ignore")
    .dropna()
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
display(correlations_with_class.rename("correlation_with_Class").to_frame())

## 9. Findings from this run

The statements below are generated from the dataset loaded above. They report simple counts, percentages, medians, and the strongest absolute numeric correlations; they do not claim model performance or causal explanations. Re-run this cell after re-running the analysis to refresh the findings.

In [ ]:
total_rows = len(df)
fraud_count = int(class_counts.loc[1])
nonfraud_count = int(class_counts.loc[0])
fraud_percentage = fraud_count / total_rows * 100 if total_rows else 0.0
missing_cells_total = int(df.isna().sum().sum())
duplicate_rows = int(df.duplicated().sum())

def median_amount_for(class_value):
    amounts = df.loc[df["Class"] == class_value, "Amount"].dropna()
    return float(amounts.median()) if not amounts.empty else None

nonfraud_median = median_amount_for(0)
fraud_median = median_amount_for(1)
if not correlations_with_class.empty:
    strongest_feature = correlations_with_class.index[0]
    strongest_correlation = float(correlations_with_class.iloc[0])
    correlation_line = (
        f"Among numeric columns other than `Class`, `{strongest_feature}` has the largest absolute "
        f"correlation with `Class` in this dataset (r = {strongest_correlation:.4f})."
    )
else:
    correlation_line = "No numeric feature correlations with `Class` could be calculated."

median_line = (
    f"Median `Amount` is {nonfraud_median:,.2f} for non-fraud and {fraud_median:,.2f} for fraud."
    if nonfraud_median is not None and fraud_median is not None
    else "A median `Amount` could not be calculated for at least one class."
)

findings = [
    f"The loaded dataset contains {total_rows:,} rows and {df.shape[1]:,} columns.",
    f"It contains {nonfraud_count:,} non-fraud rows and {fraud_count:,} fraud rows; fraud accounts for {fraud_percentage:.4f}% of all rows.",
    f"There are {missing_cells_total:,} missing cells and {duplicate_rows:,} duplicate rows.",
    median_line,
    correlation_line,
]
display(Markdown("\n".join(f"- {finding}" for finding in findings)))

## Select the Python virtual environment kernel in VS Code

1. Open the VigiLens project folder in VS Code.
2. Open this notebook and choose **Select Kernel** in the upper-right corner.
3. Choose **Python Environments**, then select the interpreter at `<project-root>/.venv/bin/python` (macOS/Linux) or `<project-root>\.venv\Scripts\python.exe` (Windows).
4. If `.venv` is not listed, use **Enter interpreter path** and browse to that Python executable.
5. Run cells from top to bottom. If a dependency is missing, ask before installing packages; the project dependencies are listed in `requirements.txt`.

The kernel metadata below names the standard Python 3 kernel. In VS Code, selecting the project `.venv` interpreter makes that virtual environment the notebook kernel.